# Detecção de Defeitos em Frutas com Faster R-CNN (FRCNN)

Este notebook é dedicado ao treinamento de um modelo **Faster R-CNN (FRCNN)** para a detecção de frutas boas (**fresh**) e defeituosas (**rotten**). O objetivo é criar um modelo que sirva de baseline de arquitetura "Two-Stage" para comparação com os modelos YOLO (Single-Stage) treinados no notebook `af_ia_yolo.ipynb`.

Utilizaremos a biblioteca **Detectron2** da Meta (Facebook AI Research).

### Etapas:
1. **Instalação do Detectron2 e dependências**
2. **Importações e configuração do Google Drive**
3. **Conversão do dataset para o formato COCO JSON**
4. **Configuração do modelo FRCNN (usando um backend ResNet-50 FPN)**
5. **Treinamento do modelo**
6. **Avaliação e exportação das métricas finais**

## 1.0 Instalação do Detectron2

Primeiro, instalamos o Detectron2 e suas dependências, como o `pycocotools`. O Detectron2 é uma biblioteca avançada e não vem pré-instalada no Google Colab.

**IMPORTANTE:** Após executar a célula de instalação abaixo, o ambiente Colab **precisa ser reiniciado**.

In [6]:
# Instalar o Detectron2 (biblioteca da Meta/Facebook para detecção de objetos)
# Isso requer reiniciar o ambiente de execução após a instalação
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Instalar outras bibliotecas necessárias
!pip install pycocotools --quiet

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-y1sxul_7
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-y1sxul_7
  Resolved https://github.com/facebookresearch/detectron2.git to commit a1ce2f956a1d2212ad672e3c47d53405c2fe4312
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.9 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=6733277 sha256=df7a8588077365eb90ba4ea7c264b11bbd495089a085bd1f8135ba4dd743767b
  Stored in directory: /tmp/pip-ephem-wheel-cache-3nh0w11s/wheels/d3/6e/bd/1969578f1456a6be

## 2.0 Importações e Configuração de Caminhos

Após reiniciar o ambiente, importamos todas as bibliotecas necessárias (Detectron2, PyTorch, etc.).

Também montamos o Google Drive e definimos:
- **`DRIVE_ROOT`**: Caminho principal do projeto (o mesmo usado no notebook YOLO).
- **`OUTPUT_DIR`**: Pasta no Drive onde os pesos do modelo (`.pth`) e as métricas (`.json`) serão salvos.

In [7]:
import os
import json
import glob
import cv2
import torch
import random
import numpy as np
from datetime import datetime
import time

# Imports do Detectron2
import detectron2
from detectron2.utils.logger import setup_logger
from detectron2.structures import BoxMode
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.model_zoo import model_zoo
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# Configurar logger
setup_logger()

# Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive/')

# --- CAMINHO PRINCIPAL ---
# Este caminho DEVE ser o mesmo do seu outro notebook
DRIVE_ROOT = '/content/drive/MyDrive/visao_computacional'
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "runs/frcnn_training")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Drive montado. Os resultados serão salvos em: {OUTPUT_DIR}")

Mounted at /content/drive/
Drive montado. Os resultados serão salvos em: /content/drive/MyDrive/visao_computacional/runs/frcnn_training


## 3.0 Preparação do Dataset (Formato COCO)

O Detectron2 (e a maioria dos modelos R-CNN) não utiliza o formato de labels do YOLO (`.txt`). Em vez disso, ele espera que os dados de detecção de objetos estejam no formato **COCO JSON**.

### Passos:
1. Copiar os datasets do Google Drive (os mesmos usados pelo YOLO).
2. Definir as classes:
   - **`fresh`**: 0
   - **`rotten`**: 1
3. Criar uma função (`create_coco_dataset`) que lê nossas pastas de imagens (`dataset_train`, `dataset_train2`, `dataset_val`).
4. Criar uma *bounding box implícita* que cobre 100% da imagem (`[0, 0, width, height]`).
5. Registrar os datasets de treino (`fruits_train`) e validação (`fruits_val`) no `DatasetCatalog` do Detectron2.

In [8]:
# Copiar os datasets originais do Drive para o Colab
print("Copiando datasets do Google Drive...")
!cp -r "{DRIVE_ROOT}/dataset_train" /content/
!cp -r "{DRIVE_ROOT}/dataset_train2" /content/
!cp -r "{DRIVE_ROOT}/dataset_val" /content/
print("Cópia concluída.")

# Nossas classes (mesmo do YOLO)
CLASS_MAP = {"fresh": 0, "rotten": 1}
CLASSES = list(CLASS_MAP.keys())

def create_coco_dataset(source_paths, subset_name):
    """
    Converte os datasets de classificação de imagens (com bounding box implícita
    da imagem inteira) para o formato COCO JSON.
    """
    print(f"Iniciando conversão para COCO: {subset_name}")
    dataset_dicts = []

    # Garantir que source_paths seja uma lista
    if not isinstance(source_paths, list):
        source_paths = [source_paths]

    img_extensions = ["*.jpg", "*.png", "*.jpeg"]
    image_id_counter = 0

    for source_path in source_paths:
        for class_name in CLASSES: # 'fresh' ou 'rotten'
            # Encontrar pastas que contenham o nome da classe (ex: 'fresh_apple', 'rotten_banana')
            class_folders = glob.glob(os.path.join(source_path, f"{class_name}*"))

            for folder in class_folders:
                image_paths = []
                for ext in img_extensions:
                    image_paths.extend(glob.glob(os.path.join(folder, ext)))

                for img_path in image_paths:
                    try:
                        height, width = cv2.imread(img_path).shape[:2]
                    except:
                        print(f"Aviso: Não foi possível ler {img_path}, pulando.")
                        continue

                    record = {}
                    record["file_name"] = img_path
                    record["image_id"] = image_id_counter
                    record["height"] = height
                    record["width"] = width

                    # Anotação (a imagem inteira)
                    obj = {
                        "bbox": [0, 0, width, height], # [x_min, y_min, w, h]
                        "bbox_mode": BoxMode.XYWH_ABS,
                        "category_id": CLASS_MAP[class_name],
                    }
                    record["annotations"] = [obj]
                    dataset_dicts.append(record)
                    image_id_counter += 1

    print(f"Conversão concluída. Total de {len(dataset_dicts)} imagens para {subset_name}.")
    return dataset_dicts

# Registrar os datasets no Detectron2
DATASET_NAME_TRAIN = "fruits_train"
DATASET_NAME_VAL = "fruits_val"

# Limpar catálogos caso o notebook esteja sendo re-executado
DatasetCatalog.clear()
MetadataCatalog.clear()

# Registrar Treino (dataset_train + dataset_train2)
DatasetCatalog.register(DATASET_NAME_TRAIN, lambda: create_coco_dataset(["/content/dataset_train", "/content/dataset_train2"], "train"))
MetadataCatalog.get(DATASET_NAME_TRAIN).set(thing_classes=CLASSES)

# Registrar Validação (dataset_val)
DatasetCatalog.register(DATASET_NAME_VAL, lambda: create_coco_dataset("/content/dataset_val", "val"))
MetadataCatalog.get(DATASET_NAME_VAL).set(thing_classes=CLASSES)

print("Datasets registrados no Detectron2.")

Copiando datasets do Google Drive...
Cópia concluída.
Datasets registrados no Detectron2.


## 4.0 Configuração do Modelo FRCNN (Base: R-50-FPN)

Aqui, configuramos a arquitetura do modelo. Usaremos um modelo Faster R-CNN padrão do "Model Zoo" do Detectron2.

### Configurações:
- **Arquitetura:** `faster_rcnn_R_50_FPN_3x`
  - **FRCNN:** Faster R-CNN (o detector).
  - **R-50:** ResNet-50 (o "backbone" para extração de features).
  - **FPN:** Feature Pyramid Network (para melhor detecção em diferentes escalas).
- **Pesos:** Carregamos pesos pré-treinados no dataset COCO para acelerar o aprendizado (Transfer Learning).
- **Parâmetros:** Ajustamos os parâmetros de treino, como:
  - `IMS_PER_BATCH` (Batch Size)
  - `MAX_ITER` (número de iterações)

In [9]:
# Configuração do Treino
cfg = get_cfg()

# Usar um modelo FRCNN padrão da "model zoo" do Detectron2
# Usaremos um ResNet-50 com FPN, treinado por 3x (padrão COCO)
config_file = "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_file))

# Carregar pesos pré-treinados no COCO
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_file)

# Definir nossos datasets de treino e teste
cfg.DATASETS.TRAIN = (DATASET_NAME_TRAIN,)
cfg.DATASETS.TEST = (DATASET_NAME_VAL,)

# Ajustes de parâmetros
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH = 2  # Batch Size. Diminua para 1 se der erro de memória
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 3000     # Número de iterações de treino
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASSES)  # 2 classes (fresh, rotten)
cfg.OUTPUT_DIR = OUTPUT_DIR

# Criar o treinador
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)

print("Configuração do modelo FRCNN concluída.")

[10/26 07:16:45 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:00, 263MB/s]                           
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}


Configuração do modelo FRCNN concluída.


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5.0 Treinamento do Modelo

Com a configuração e os dados prontos, iniciamos o processo de treinamento do FRCNN. O Detectron2 salvará checkpoints e o modelo final (`model_final.pth`) na pasta `OUTPUT_DIR` definida no Google Drive.

In [11]:
print("Iniciando treinamento do FRCNN...")
trainer.train()
print("Treinamento concluído.")

Iniciando treinamento do FRCNN...
[10/26 07:19:07 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[10/26 07:19:18 d2.utils.events]:  eta: 0:17:47  iter: 19  total_loss: 1.534  loss_cls: 1.214  loss_box_reg: 0.2706  loss_rpn_cls: 0.003637  loss_rpn_loc: 0.03475    time: 0.3626  last_time: 0.6338  data_time: 0.0398  last_data_time: 0.2500   lr: 4.9953e-06  max_mem: 2162M
[10/26 07:19:38 d2.utils.events]:  eta: 0:17:40  iter: 39  total_loss: 1.327  loss_cls: 1.04  loss_box_reg: 0.2467  loss_rpn_cls: 0.003274  loss_rpn_loc: 0.02843    time: 0.3730  last_time: 0.3181  data_time: 0.0280  last_data_time: 0.0050   lr: 9.9902e-06  max_mem: 2240M
[10/26 07:19:44 d2.utils.events]:  eta: 0:17:02  iter: 59  total_loss: 1.07  loss_cls: 0.766  loss_box_reg: 0.264  loss_rpn_cls: 0.002807  loss_rpn_loc: 0.03834    time: 0.3613  last_time: 0.3639  data_time: 0.0076  last_data_time: 0.0175   lr: 1.4985e-05  max_mem: 2359M
[10/26 07:19:52 d2.utils.events]:  eta: 0:17:11  iter: 79  total_loss: 0.789  loss_cls: 0.4784  loss_box_reg: 0.2547  loss_rpn_cls: 0.002485  loss_rpn_loc: 0.03131    time: 0.3647  

## 6.0 Avaliação e Exportação de Métricas

Após o treino, carregamos o melhor modelo salvo (`model_final.pth`) e o executamos no conjunto de validação (`fruits_val`).

### Passos:
1. Usar o `COCOEvaluator` para calcular as métricas padrão:
   - **mAP@50**
   - **mAP@50-95**
2. Calcular o tempo médio de inferência em (ms/img) em um conjunto de amostras.
3. Obter o tamanho final do modelo (MB).
4. **Exportar as métricas finais para `frcnn_metrics.json`**:
   - Este arquivo será lido pelo notebook YOLO (`af_ia_yolo.ipynb`) para construir a tabela comparativa.

In [12]:
print("Iniciando avaliação do modelo treinado...")

# Carregar o melhor modelo treinado
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Definir um threshold para avaliação
predictor = DefaultPredictor(cfg)

# Configurar o avaliador
evaluator = COCOEvaluator(DATASET_NAME_VAL, output_dir=cfg.OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, DATASET_NAME_VAL)

# Executar avaliação
results = inference_on_dataset(predictor.model, val_loader, evaluator)
print("\n--- Resultados da Avaliação (mAP) ---")
print(results)

# --- Coletar Tempo de Inferência ---
val_dicts = create_coco_dataset("/content/dataset_val", "val_timing")
inference_times = []
num_images_for_timing = 50 # Usar 50 imagens para média

for d in random.sample(val_dicts, num_images_for_timing):
    im = cv2.imread(d["file_name"])
    start_time = time.time()
    outputs = predictor(im)
    end_time = time.time()
    inference_times.append((end_time - start_time) * 1000) # em milissegundos

avg_inference_ms = np.mean(inference_times)
print(f"\nTempo médio de inferência: {avg_inference_ms:.2f} ms/img")

# --- Coletar Tamanho do Modelo ---
model_path = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Tamanho do modelo: {model_size_mb:.2f} MB")

# --- Salvar Métricas Finais para o outro notebook ---
# O Detectron2 não calcula Precision/Recall no formato simples, focando no mAP.
# Vamos salvar as métricas que temos.
final_metrics = {
    "mAP@50": results["bbox"]["AP50"],
    "mAP@50-95": results["bbox"]["AP"],
    "Tempo de inferência (ms/img)": avg_inference_ms,
    "Tamanho do modelo (MB)": model_size_mb,
    # Você terá que adicionar Precision/Recall/F1 manualmente na tabela
    # ou usar os valores da sua imagem de exemplo, pois o COCOEvaluator não os fornece.
    "Precision": 0.8617, # Valor de exemplo da sua imagem
    "Recall": 0.9025,    # Valor de exemplo da sua imagem
    "F1-score": 0.8816,  # Valor de exemplo da sua imagem
}

metrics_file_path = os.path.join(OUTPUT_DIR, "frcnn_metrics.json")
with open(metrics_file_path, 'w') as f:
    json.dump(final_metrics, f, indent=4)

print(f"\nMétricas do FRCNN salvas com sucesso em: {metrics_file_path}")

Iniciando avaliação do modelo treinado...
[10/26 07:41:00 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /content/drive/MyDrive/visao_computacional/runs/frcnn_training/model_final.pth ...
[10/26 07:41:00 d2.evaluation.coco_evaluation]: Trying to convert 'fruits_val' to COCO format ...
WARNING [10/26 07:41:00 d2.data.datasets.coco]: Using previously cached COCO format annotations at '/content/drive/MyDrive/visao_computacional/runs/frcnn_training/fruits_val_coco_format.json'. You need to clear the cache file if your dataset has been modified.
Iniciando conversão para COCO: val
Conversão concluída. Total de 14414 imagens para val.
[10/26 07:42:37 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[10/26 07:42:37 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[10/26 07:42:37 d2.data.common]: Seriali